# Chapter 9.1 - Working with Sequences

A sequence is an ordered collection whose position matters. This notebook turns a time series into supervised examples, trains a small autoregressive predictor, and separates one-step prediction from recursive forecasting.

## How to use this notebook

Run the notebook from top to bottom in a clean kernel. Everything is generated from small tensors or inline text, so there are no downloads. Before important cells, predict the time, batch, feature, vocabulary, and hidden-state shapes. Treat every assertion as an executable contract rather than decoration.

## You are done when you can

- explain why ordinary shuffled rows do not fully describe sequence prediction
- build lagged features and correctly aligned labels
- split time series without leaking the future into training
- distinguish one-step predictions from recursive multi-step forecasts
- diagnose an off-by-one target bug


In [ ]:
import math
import random
from collections import Counter

import torch
from torch import nn
from torch.nn import functional as F

torch.manual_seed(0)
random.seed(0)
torch.set_printoptions(precision=4, sci_mode=False)

def shape(x):
    return tuple(x.shape)


## 9.1.0 The Problem This Notebook Solves

In ordinary tabular regression, each row can often be interpreted independently. In a sequence, the observation at time `t` is related to earlier observations. A temperature, price, sensor reading, or token arrives in an order.

An **autoregressive model** predicts a future value using previous values from the same sequence. If the lag length is `tau`, its input at time `t` is

```text
[x[t - tau], ..., x[t - 2], x[t - 1]]
```

and its target is `x[t]`. The word *lag* means a backward time offset. The immediate engineering problem is therefore not merely choosing a network. It is constructing examples without breaking time alignment.


## 9.1.1 Autoregressive Models: Turn History Into Features

We first create a smooth synthetic signal with noise. Then `make_lagged` slides a window across it. One input row contains `tau` consecutive past values; the corresponding label is the very next value.

Before running the cell, predict the shapes when a length-12 sequence uses `tau=4`. There are only `12 - 4 = 8` complete input-target pairs.


In [ ]:
def make_lagged(series, tau):
    if series.ndim != 1:
        raise ValueError("series must be one-dimensional")
    if not 1 <= tau < len(series):
        raise ValueError("tau must be between 1 and len(series) - 1")
    features = torch.stack([series[i : i + tau] for i in range(len(series) - tau)])
    labels = series[tau:].reshape(-1, 1)
    return features, labels

tiny = torch.arange(12, dtype=torch.float32)
X_tiny, y_tiny = make_lagged(tiny, tau=4)
print(X_tiny[:3])
print(y_tiny[:3])
assert shape(X_tiny) == (8, 4)
assert shape(y_tiny) == (8, 1)
assert torch.equal(X_tiny[0], torch.tensor([0.0, 1.0, 2.0, 3.0]))
assert y_tiny[0].item() == 4.0


## 9.1.2 Sequence Models and Chronological Splits

A **time series** is a sequence indexed by time. A **univariate** time series has one measured value at each time; a multivariate series has several. The lagged table above is a fixed-window sequence model: it assumes the last `tau` values contain the useful context.

Validation must imitate deployment. If training randomly samples windows from the whole timeline, a window from late in the series can influence the model before we evaluate on an earlier window. A **chronological split** trains on the past and evaluates on a later contiguous region.

The model below is an MLP, not yet an RNN. That is intentional: it isolates the data contract shared by sequence predictors.


In [ ]:
time = torch.arange(0, 120, dtype=torch.float32)
series = torch.sin(time * 0.12) + 0.15 * torch.randn(120)
tau = 8
features, labels = make_lagged(series, tau)

split = 84
X_train, y_train = features[:split], labels[:split]
X_valid, y_valid = features[split:], labels[split:]
model = nn.Sequential(nn.Linear(tau, 16), nn.ReLU(), nn.Linear(16, 1))

assert X_train[-1, -1].item() == series[split + tau - 2].item()
assert y_train[-1].item() == series[split + tau - 1].item()
assert X_valid[0, 0].item() == series[split].item()
assert shape(model(X_train[:3])) == (3, 1)


## 9.1.3 Training: What Repeats and What Changes

One **epoch** is one complete optimization pass over the chosen training examples. In each epoch:

1. the current parameters produce predictions;
2. mean squared error measures prediction error;
3. `backward()` stores gradients in parameter `.grad` fields;
4. `optimizer.step()` changes parameters;
5. `zero_grad()` clears old gradients before the next pass.

The data stay fixed here. Model parameters and optimizer state change. Full-batch training keeps the control flow visible.


In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.03)
for epoch in range(80):
    optimizer.zero_grad()
    predictions = model(X_train)
    loss = F.mse_loss(predictions, y_train)
    loss.backward()
    optimizer.step()

model.eval()
with torch.no_grad():
    train_mse = F.mse_loss(model(X_train), y_train).item()
    valid_mse = F.mse_loss(model(X_valid), y_valid).item()

print({"train_mse": train_mse, "valid_mse": valid_mse})
assert math.isfinite(train_mse)
assert math.isfinite(valid_mse)


## 9.1.4 Prediction: One Step Versus Many Steps

A **one-step prediction** gets the real previous values at every time. A **recursive forecast** predicts one value, appends that prediction to its own history, and uses it to predict again. Recursive errors can compound because later inputs contain earlier mistakes.

The cell starts both methods at the validation boundary. The one-step path uses `X_valid`, which was made from observed values. The recursive path receives only the last real training window and then feeds itself.


In [ ]:
with torch.no_grad():
    one_step = model(X_valid).squeeze(1)
    history = X_valid[0].clone()
    recursive_values = []
    for _ in range(len(X_valid)):
        next_value = model(history.reshape(1, -1)).squeeze()
        recursive_values.append(next_value)
        history = torch.cat((history[1:], next_value.reshape(1)))
    recursive = torch.stack(recursive_values)

print("first five one-step:", one_step[:5])
print("first five recursive:", recursive[:5])
assert shape(one_step) == shape(recursive) == (len(X_valid),)
assert torch.allclose(one_step[0], recursive[0])


## 9.1.5 Break It Deliberately: An Off-by-One Label

If labels use the final value *inside* each input window, the model is rewarded for copying a value it already received. The shapes still look correct, so shape checks alone cannot catch this bug. Alignment needs a value-level contract.


In [ ]:
wrong_labels = series[tau - 1 : -1].reshape(-1, 1)
print("input window:", features[0])
print("wrong label:", wrong_labels[0].item())
print("correct label:", labels[0].item())

assert shape(wrong_labels) == shape(labels)
assert wrong_labels[0].item() == features[0, -1].item()
assert labels[0].item() == series[tau].item()
assert wrong_labels[0].item() != labels[0].item()


## 9.1 Checkpoint

Answer these without rerunning the notebook. Short markdown answers are enough.

1. For a sequence of length N and lag tau, how many complete examples exist?
2. Why can a random train-validation split leak temporal information?
3. What information differs between one-step and recursive prediction?
4. Why does correct shape not prove correct label alignment?
5. Which objects change during the training loop?
